In [ ]:
import torch
import pandas as pd
import os
import warnings
from collections.abc import Iterable
from utils_metrics import *
from utils_sampling_processing import process_batch, process_data, sampling
from model import compute_loss
from run_training import define_model

# Samples vs Full Graph rel breakdown

In [ ]:
data = torch.load('../data_trimmed/CV1/data.pt', weights_only=False)

In [ ]:
data.num_edges

In [ ]:
for et in data.edge_types:
    print(f'{et}: \t{data[et].num_edges}')

In [ ]:
def edge_counts(data):
    counts = {}
    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] in processed_edge_types:
            continue
        else:
            processed_edge_types.add(edge_type[1])
            counts[edge_type[1]] = data[edge_type].num_edges

    return counts


In [ ]:
counts_full = edge_counts(data)
edge_series = pd.Series(counts_full)
distribution_full = (edge_series / edge_series.sum()) * 100

print(distribution_full.sort_values(ascending=False))

In [ ]:
from utils_sampling_processing import sampling

def batch_edge_means(data, sampling_layers, batch_size, num_samples):
    config = {
        'sampling_layers': sampling_layers,
        'batch_size': batch_size,
        'num_samples': num_samples,
        'seeded_type': 'disease'
    }

    loader = sampling(config, data)
    batch_list_counts = []
    for batch in iter(loader):
        # print(batch.num_edges)
        batch_list_counts.append(edge_counts(batch))

    batches = pd.DataFrame(batch_list_counts).T
    batches = batches / batches.sum(axis=0) * 100
    return batches.mean(axis=1).sort_values(ascending=False)

In [ ]:
data_pred = torch.load('../data_trimmed/hetero_graph_AT.pt', weights_only=False)
data_pred['disease']['input_nodes'] = torch.zeros(data_pred['disease'].num_nodes, dtype=torch.bool)
seeded_nodes = data_pred[('drug', 'indication', 'disease')]['edge_index'][1].unique()
data_pred['disease']['input_nodes'][seeded_nodes] = True

In [ ]:
counts_full = edge_counts(data_pred)
edge_series = pd.Series(counts_full)
distribution_full_pred = (edge_series / edge_series.sum()) * 100

print(distribution_full_pred.sort_values(ascending=False))

In [ ]:
print(batch_edge_means(data_pred, 3, 342, 512))

In [ ]:
print(batch_edge_means(data, 3, 342, 512))

In [ ]:
print(batch_edge_means(data, 3, 682, 744) - batch_edge_means(data_pred, 3, 682, 744))

# Looking into overlap edges between sampled graphs

### I want to know, which ones are not exclusive to each batch

In [ ]:
from collections import Counter

def exclusivity_report(loader, data):
    batches_e_ids = {}

    for batch_id, batch in enumerate(iter(loader)):
        batches_e_ids[batch_id] = {}
        print(f'Batch {batch_id}: {batch.num_edges} edges')
        for et in batch.edge_types:
            if et[1] not in batches_e_ids[batch_id]:
                # print(f'\tEdge type: {et[1]}, Number of edges: {batch[et].num_edges}')
                batches_e_ids[batch_id][et[1]] = set(batch[et].e_id.tolist())
                # batches_e_ids[batch_id][et].update(batch[et].e_id.tolist())
                # print(f"{et[1]}: {len(set(batch[et].e_id.tolist()))}")
            else:
                continue

    print()
    # for et in batch.edge_types:
    #     print(f'{et[1]}: \t{len(all_edges[et[1]]) / data[et].num_edges * 100:.2f}%')
    # print()

    exclusivity_report = {}
    # for et in batches_e_ids[0].keys():
    processed_edge_types = set()
    for edge_type in data.edge_types:
        et = edge_type[1]
        if et in processed_edge_types:
            # print(f"{et}: {len(set.union(*[batches_e_ids[b_id][et] for b_id in batches_e_ids]))} unique edges across all batches")
            continue
        else:
            processed_edge_types.add(et)
            # print(f'Edge type: {et}')
            # 1. Flatten all IDs for this edge type and count occurrences
            all_ids = [eid for b_id in batches_e_ids for eid in batches_e_ids[b_id][et]]
            counts = Counter(all_ids)
            print(f'{et}: \t{len(counts)} ({len(counts) / data[edge_type].num_edges * 100:.2f})%')

            # 2. Identify the IDs that only appeared exactly once across all batches
            exclusive_pool = {eid for eid, count in counts.items() if count == 1}

            # 3. Find how many exclusive IDs belong to each batch
            for b_id in batches_e_ids:
                if b_id not in exclusivity_report:
                    exclusivity_report[b_id] = {}
                # Intersection of the batch set and the global exclusive pool
                batch_exclusives = batches_e_ids[b_id][et].intersection(exclusive_pool)
                exclusivity_report[b_id][et] = round(len(batch_exclusives) / len(batches_e_ids[b_id][et]) * 100, 2)

    print()
    return pd.DataFrame(exclusivity_report)

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 3, 'batch_size': 342, 'num_samples': 512, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 512, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 744, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 1200, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 6, 'batch_size': 682, 'num_samples': 1200, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 5200, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 10, 'batch_size': 682, 'num_samples': 10000, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 512, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 744, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 1200, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 5200, 'seeded_type': 'disease'}, data)).mean(axis=1))

### edge counts

i think the functions are repeated from above

In [ ]:
def edge_counts(data):
    counts = {}
    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] in processed_edge_types:
            continue
        else:
            processed_edge_types.add(edge_type[1])
            counts[edge_type[1]] = data[edge_type].num_edges

    return counts

def batch_edge_means(data, sampling_layers, batch_size, num_samples):
    config = {
        'sampling_layers': sampling_layers,
        'batch_size': batch_size,
        'num_samples': num_samples,
        'seeded_type': 'disease'
    }

    loader = sampling(config, data)
    batch_list_counts = []
    for batch in iter(loader):
        # print(batch.num_edges)
        batch_list_counts.append(edge_counts(batch))

    batches = pd.DataFrame(batch_list_counts).T
    batches = batches / batches.sum(axis=0) * 100
    return batches.mean(axis=1).sort_values(ascending=False)


In [ ]:
loader = sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 744, 'seeded_type': 'disease'}, data)
batch = next(iter(loader))
print(batch.num_edges)

edge_counts(batch)

In [ ]:
print(batch_edge_means(data, 5, 682, 744))

In [ ]:
print(batch_edge_means(data, 5, 682, 1200))

In [ ]:
print(batch_edge_means(data, 5, 682, 5200))

### seeding drugs instead

In [ ]:
data['disease'].num_nodes, data['disease']['input_nodes'].sum()

In [ ]:
data['drug']['input_nodes'] = torch.zeros(data['drug'].num_nodes, dtype=torch.bool)
seeded_nodes = data[('drug', 'indication', 'disease')]['edge_index'][0].unique()
data['drug']['input_nodes'][seeded_nodes] = True

In [ ]:
data['drug'].num_nodes, data['drug']['input_nodes'].sum()

In [ ]:
1801 // 2 + 1

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 452, 'num_samples': 744, 'seeded_type': 'drug'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 901, 'num_samples': 744, 'seeded_type': 'drug'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 901, 'num_samples': 5200, 'seeded_type': 'drug'}, data)).mean(axis=1))

# Coverage over epochs

In [ ]:
def coverage_epochs(data, sampling_layers, batch_size, num_samples, epochs=300):
    batches_e_ids = {}

    processed_edge_types = set()
    for edge_type in data.edge_types:
        et = edge_type[1]
        if et in processed_edge_types:
            continue
        else:
            processed_edge_types.add(et)
            batches_e_ids[et] = set()

    loader = sampling({'sampling_layers': sampling_layers, 'batch_size': batch_size, 'num_samples': num_samples, 'seeded_type': 'disease'}, data)

    for epoch in range(epochs):
        if epoch % 10 == 0:
            print(f'Epoch {epoch}/{epochs}')

        for batch_id, batch in enumerate(iter(loader)):
            # batches_e_ids[batch_id] = {}
            # print(f'\nBatch {batch_id}: {batch.num_edges} edges')
            processed_edge_types = set()
            for et in batch.edge_types:
                if et[1] in processed_edge_types:
                    continue
                else:
                    batches_e_ids[et[1]].update(batch[et].e_id.tolist())
                    # batches_e_ids[batch_id][et].update(batch[et].e_id.tolist())
                    # print(f"{et[1]}: {len(set(batch[et].e_id.tolist()))}")

    print()

    processed_edge_types = set()
    for edge_type in data.edge_types:
        et = edge_type[1]
        if et in processed_edge_types:
            continue
        else:
            processed_edge_types.add(et)
            print(f'{et}: \t{len(batches_e_ids[et])} ({len(batches_e_ids[et]) / data[edge_type].num_edges * 100:.2f}%)')

    return batches_e_ids


In [ ]:
coverage_epochs(data, 3, 342, 512, epochs=100)

In [ ]:
coverage_epochs(data, 3, 342, 512, epochs=300)

In [ ]:
coverage_epochs(data, 3, 342, 744, epochs=300)

In [ ]:
coverage_epochs(data, 3, 682, 744, epochs=300)

In [ ]:
coverage_epochs(data, 4, 682, 596, epochs=300)

In [ ]:
coverage_epochs(data, 5, 682, 512, epochs=300)

In [ ]:
coverage_epochs(data, 5, 682, 744, epochs=300)

In [ ]:
coverage_epochs(data, 5, 682, 1200, epochs=300)

In [ ]:
coverage_epochs(data, 5, 682, 5200, epochs=300)

In [ ]:
coverage_epochs(data, 10, 682, 5200, epochs=300)

In [ ]:
coverage_epochs(data, 5, 1363, 512, epochs=300)

In [ ]:
coverage_epochs(data, 5, 1363, 744, epochs=300)

In [ ]:
coverage_epochs(data, 5, 1363, 1200, epochs=300)

In [ ]:
coverage_epochs(data, 5, 1363, 5200, epochs=300)

In [ ]:
coverage_epochs(data, 10, 1363, 5200, epochs=300)

In [ ]:
import math

print("{:.2e}".format(math.comb(1363, 682)))

In [ ]:
sampled_dict = coverage_epochs(data, 5, 682, 5200, epochs=300)

In [ ]:
sampled_dict

In [ ]:
def add_sampled_mask(data, sampled_dict):
    """
    Adds a .sampled boolean tensor to each edge_type in HeteroData
    based on the indices provided in sampled_dict.
    """
    for edge_type in data.edge_types:
        num_edges = data[edge_type].num_edges
        mask = torch.zeros(num_edges, dtype=torch.bool)

        sampled_indices = sampled_dict[edge_type[1]]

        if sampled_indices is not None and len(sampled_indices) > 0:
            idx_tensor = torch.tensor(list(sampled_indices), dtype=torch.long)
            mask[idx_tensor] = True

        data[edge_type].sampled = mask

    return data

data = add_sampled_mask(data, sampled_dict)

In [ ]:
orocessed_edge_types = set()
for edge_type in data.edge_types:
    if edge_type[1] in orocessed_edge_types:
        continue
    else:
        sampled_count = data[edge_type].sampled.sum()
        total_count = data[edge_type].num_edges
        print(f"{edge_type[1]}: {sampled_count} ({sampled_count / total_count:.2%})")
        orocessed_edge_types.add(edge_type[1])



In [ ]:
sum(data[edge_type].sampled.tolist())

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time
import statistics

def get_strictly_sorted_shell_layout(G):
    # 1. Calculate centrality (Degree centrality is fastest for high node counts)
    # Using degree centrality because it's O(E), betweenness is O(VE)
    centrality = nx.degree_centrality(G)
    
    # 2. Sort nodes by centrality (highest first)
    sorted_nodes = [n for n, c in sorted(centrality.items(), key=lambda item: item[1], reverse=True)]
    
    # 3. Define the shells (The "nlist")
    # We want a small center, a medium ring, and a large outer ring
    num_nodes = len(sorted_nodes)
    
    # Tiered thresholds (adjust percentages to change visual density)
    inner_limit = max(1, int(num_nodes * 0.05))   # Top 5%
    middle_limit = max(1, int(num_nodes * 0.25))  # Next 25%
    
    shells = [
        sorted_nodes[:inner_limit],                # Inner circle (Hubs/Bottlenecks)
        sorted_nodes[inner_limit:middle_limit],   # Middle ring
        sorted_nodes[middle_limit:]               # Outer ring (Peripheral nodes)
    ]
    
    # 4. Generate the layout using our custom shell list
    pos = nx.shell_layout(G, nlist=shells)
    return pos

def plot_edge_type_analysis(data, layout = 'shell', threshold_coverage=90):
# 1. Create a consistent color map for all node types in the graph
    node_types = data.node_types
    colors = cm.rainbow(np.linspace(0, 1, len(node_types)))
    type_to_color = {nt: colors[i] for i, nt in enumerate(node_types)}
    
    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] in processed_edge_types:
        # if edge_type != ('drug', 'drug_drug', 'drug'):
            continue
        else:
            src_type, rel, dst_type = edge_type
            processed_edge_types.add(rel)
            start_time = time.time()

            print(f"\nAnalyzing edge type: {edge_type[1]}")
            # 1. Extract the specific edge type
            # edge_type is a triplet like ('node_a', 'relates_to', 'node_b')
            edge_index = data[edge_type].edge_index
            sampled_mask = data[edge_type].sampled.tolist()
            coverage = sum(sampled_mask) / len(sampled_mask) * 100
            if threshold_coverage < coverage:
                print(f"Skipping {edge_type[1]} due to high coverage ({coverage:.2f}%)")
                continue
            print(f"Coverage: {coverage:.2f}%")

            # 2. Convert to NetworkX (treating as undirected for GCC/Bottleneck analysis)
            # print('converting to networkx graph...')
            G = nx.Graph()

            if coverage < 90:
                G_sampled = nx.Graph()

            # Add edges (using .tolist() for speed in small/medium graphs)
            # edges = edge_index.t().tolist()
            # G.add_edges_from(edges)

            src_nodes = edge_index[0].tolist()
            dst_nodes = edge_index[1].tolist()

            # Create unique string IDs
            # edges_with_types = [
            #     (f"{src_type}_{s}", f"{dst_type}_{d}") 
            #     for s, d in zip(src_nodes, dst_nodes)
            # ]
            # G.add_edges_from(edges_with_types)

            # Add edges one by one to store the 'sampled' status
            sampled_connected_nodes = set()
            for i in range(len(src_nodes)):
                u, v = f"{src_type}_{src_nodes[i]}", f"{dst_type}_{dst_nodes[i]}"
                G.add_edge(u, v, sampled=sampled_mask[i])

                if coverage < 90 and sampled_mask[i]:
                    G_sampled.add_edge(u, v)

                is_sampled = sampled_mask[i]
                if is_sampled:
                    sampled_connected_nodes.add(u)
                    sampled_connected_nodes.add(v)

            # 3. Identify Giant Connected Component (GCC)
            # print('identifying giant connected component...')
            components = sorted(nx.connected_components(G), key=len, reverse=True)
            # gcc_nodes = components[0]
            # G_gcc = G.subgraph(gcc_nodes).copy()

            for i in range(len(components)):
                print(f"\tComponent {i}: {len(components[i])} ({len(components[i]) / len(G.nodes()) * 100:.2f}%) nodes")

            # 4. Detect Bottlenecks (Betweenness Centrality)
            # Higher score = node lies on more shortest paths
            # print('calculating betweenness centrality (bottlenecks)...')
            centrality = nx.betweenness_centrality(G, k=min(100, len(G.nodes())), seed=42)  # Approximate for speed on larger graphs

            # 4. Determine color for each node based on its prefix
            node_colors = [type_to_color[n.split('_')[0]] for n in G.nodes()]
            node_sizes = [v * 5000 + .5 for v in centrality.values()]

            # edge_alphas = [0.1 if G[u][v]['sampled'] else 0.8 for u, v in G.edges()]
            if coverage < 10:
                edge_alphas = [0.8 if G[u][v]['sampled'] else 0.05 for u, v in G.edges()] #.5
            else:
                edge_alphas = [0.1 if G[u][v]['sampled'] else 0.8 for u, v in G.edges()]

            # edge_colors = ['red' if G[u][v]['sampled'] else 'black' for u, v in G.edges()]
            # Highlight Logic: Thick Gold border for sampled nodes, None for others
            if coverage > 90:
                edge_colors = ['red' if G[u][v]['sampled'] else 'black' for u, v in G.edges()]
                node_linewidths = [0 for n in G.nodes()]
            else: # elif coverage > 10:
                edge_colors = ['red' if G[u][v]['sampled'] else 'white' for u, v in G.edges()]
                # node_border_colors = ['black' if n in sampled_connected_nodes else 'gold' for n in G.nodes()]
                # node_linewidths = [0 if n in sampled_connected_nodes else 3.0 for n in G.nodes()]

                deg_cent_sampled = dict(nx.degree_centrality(G_sampled))
                deg_cent = dict(nx.degree_centrality(G))
                deg_sampled = dict(nx.degree(G_sampled))
                deg = dict(nx.degree(G))
                # print('sampled', min(deg_sampled.values()), max(deg_sampled.values()))
                # print('all', min(deg.values()), max(deg.values()))
                # node_linewidths = [deg_sampled.get(n, 0) * 1000 for n in G.nodes()]
                # node_border_colors = ['none' if deg[n] > deg_sampled.get(n, 0) else 'gold' for n in G.nodes()]
                # node_linewidths = [deg[n] / 2 - deg_sampled.get(n, 0) + 1 for n in G.nodes()]
                node_linewidths = []
                node_border_colors = []
                unsampled_deg = []
                count = 0
                for n in G.nodes():
                    gap_cent = deg[n] / 2 - deg_sampled.get(n, 0)
                    # gap_cent = deg_cent[n] - deg_cent_sampled.get(n, 0)
                    if gap_cent > 0:
                        # node_linewidths.append(deg_cent[n] - deg_cent_sampled.get(n, 0) * 100 + 2)
                        node_linewidths.append(10)
                        count += 1
                        # unsampled_deg.append(deg[n] - deg_sampled.get(n, 0))
                        unsampled_deg.append(deg_sampled.get(n, 0))
                        if deg[n] / 4 > deg_sampled.get(n, 0):
                            node_border_colors.append('gold')
                        elif deg[n] / 3 > deg_sampled.get(n, 0):
                            node_border_colors.append('lime')
                        elif deg[n] / 2 > deg_sampled.get(n, 0):
                            node_border_colors.append('magenta')
                    else:
                        node_linewidths.append(1)
                        node_border_colors.append('black')
                print(f"undersampled nodes: {count} ({count / len(G.nodes()) * 100:.2f}%)")
                print(f"unsampled degrees: min = {min(unsampled_deg):.4f}, median = {statistics.median(unsampled_deg):.4f}, max = {max(unsampled_deg):.4f}")

            # 5. Visualization
            # print('visualizing graph...')
            plt.figure(figsize=(10, 8))
            # pos = nx.shell_layout(G) # Much faster for high node counts
            if layout == 'spring':
                pos = nx.spring_layout(G, k=0.15, seed=42)
            elif layout == 'shell':
                pos = get_strictly_sorted_shell_layout(G)

            # print('drawing graph...')
            print(f"Time: {time.time() - start_time:.2f} seconds")
            nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, linewidths=node_linewidths, edgecolors=node_border_colors, alpha=0.7)
            nx.draw_networkx_edges(G, pos, alpha=edge_alphas, edge_color=edge_colors)#, width=edge_widths)
            
            plt.title(f"Analysis: {edge_type}\nGCC Size: {len(G.nodes())} nodes | Large nodes = Bottlenecks")
            plt.axis('off')
            plt.show()

# Run the analysis
plot_edge_type_analysis(data)

In [ ]:
data = add_sampled_mask(data, coverage_epochs(data, 5, 682, 1200, epochs=300))
plot_edge_type_analysis(data, threshold_coverage=95)

In [ ]:
data = add_sampled_mask(data, coverage_epochs(data, 5, 682, 1200, epochs=300))
plot_edge_type_analysis(data, threshold_coverage=95)

In [ ]:
data = add_sampled_mask(data, coverage_epochs(data, 5, 682, 1200, epochs=500))
plot_edge_type_analysis(data, threshold_coverage=95)

In [ ]:
data = add_sampled_mask(data, coverage_epochs(data, 5, 682, 1200, epochs=300))

In [ ]:
def sampled_mask(data):

    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] in processed_edge_types:
            continue
        else:
            processed_edge_types.add(edge_type[1])
            sampled = data[edge_type].sampled
            mask = data[edge_type].mask
            print(f"{edge_type[1]}: {data[edge_type].num_edges} total edges ({sampled.sum()} sampled = {sampled.sum() / data[edge_type].num_edges:.2%}%)")
            # print(f"train: {((data[edge_type].mask == 0).sum()) / data[edge_type].num_edges:.2%} edges, val: {(data[edge_type].mask == 1).sum() / data[edge_type].num_edges:.2%} edges, test: {((data[edge_type].mask == 2).sum()) / data[edge_type].num_edges:.2%} edges")

            train = val = test = 0
            for i in range(len(sampled)):
                if sampled[i] and mask[i] == 0:
                    # print(f"Sampled edge {i} is in the training set.")
                    train += 1
                elif sampled[i] and mask[i] == 1:
                    # print(f"Sampled edge {i} is in the validation set.")
                    val += 1
                elif sampled[i] and mask[i] == 2:
                    # print(f"Sampled edge {i} is in the test set.")
                    test += 1

            print(f"Sampled: Train: {train} ({train / sampled.sum():.2%}), Val: {val} ({val / sampled.sum():.2%}), Test: {test} ({test / sampled.sum():.2%})\n")

            # sampled_count = data[edge_type].sampled.sum()
            # total_count = data[edge_type].num_edges
            # print(f"{edge_type[1]}: {sampled_count} ({sampled_count / total_count:.2%})")

sampled_mask(data)

In [ ]:
data2 = torch.load('../data_trimmed/CV2/data.pt', weights_only=False)
data2 = add_sampled_mask(data2, coverage_epochs(data2, 5, 682, 1200, epochs=300))
sampled_mask(data2)

# Coverage and exclusivity of pred graph

In [ ]:
from torch_geometric.loader import HGTLoader

def sampling(config, data, workers = 0):
    loader = HGTLoader(data, 
                   num_samples=[config['num_samples']] * config['sampling_layers'], 
                   batch_size=config['batch_size'], 
                #    input_nodes=(config['node_type1'], data[config['node_type1']]['input_nodes']), 
                   input_nodes=(config['seeded_type'], data[config['seeded_type']].input_nodes if 'input_nodes' in data[config['seeded_type']] else None), 
                   shuffle=True, 
                   num_workers = workers, 
                   persistent_workers = bool(workers)
                   )
    return loader


In [ ]:
data_pred = torch.load('../data_trimmed/hetero_graph_iAT.pt', weights_only=False)
coverage_epochs(data_pred, 5, 682, 744, epochs=100)

In [ ]:
coverage_epochs(data_pred, 5, 682, 744, epochs=200)

In [ ]:
coverage_epochs(data_pred, 5, 682, 1200, epochs=100)

In [ ]:
coverage_epochs(data_pred, 5, 682, 1200, epochs=200)

In [ ]:
coverage_epochs(data_pred, 5, 682, 5200, epochs=100)

In [ ]:
coverage_epochs(data_pred, 5, 682, 5200, epochs=200)

In [ ]:
data_pred = torch.load('../data_trimmed/hetero_graph_iAT.pt', weights_only=False)
coverage_epochs(data_pred, 5, 1363, 744, epochs=100)

In [ ]:
coverage_epochs(data_pred, 5, 1363, 1200, epochs=100)

In [ ]:
coverage_epochs(data_pred, 5, 1363, 5200, epochs=100)

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 744, 'seeded_type': 'disease'}, data_pred), data_pred).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 1200, 'seeded_type': 'disease'}, data_pred), data_pred).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 5200, 'seeded_type': 'disease'}, data_pred), data_pred).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 744, 'seeded_type': 'disease'}, data_pred), data_pred).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 1200, 'seeded_type': 'disease'}, data_pred), data_pred).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 1363, 'num_samples': 5200, 'seeded_type': 'disease'}, data_pred), data_pred).mean(axis=1))

In [ ]:
data_pred = torch.load('../data_trimmed/hetero_graph_AT.pt', weights_only=False)
data_pred = add_sampled_mask(data_pred, coverage_epochs(data_pred, 5, 682, 1200, epochs=300))
sampled_mask(data_pred)

### Node coverage and by type neighbors

In [ ]:
def node_counts(data):
    counts = {}
    # processed_edge_types = set()
    for node_type in data.node_types:
        # if node_type in processed_edge_types:
        #     continue
        # else:
        #     processed_edge_types.add(node_type)
            counts[node_type] = data[node_type].num_nodes

    return counts

node_counts(data)

In [ ]:
data['disease'].input_nodes.sum(), data['drug']['input_nodes'].sum()

In [ ]:
6885 / 19101

In [ ]:
def batch_node_means(data, sampling_layers, batch_size, num_samples):
    config = {
        'sampling_layers': sampling_layers,
        'batch_size': batch_size,
        'num_samples': num_samples,
        'seeded_type': 'disease'
    }

    loader = sampling(config, data)
    batch_list_counts = []
    for batch in iter(loader):
        # print(batch.num_edges)
        batch_list_counts.append(node_counts(batch))

    batches = pd.DataFrame(batch_list_counts).T
    return batches
    # batches = batches / batches.sum(axis=0) * 100
    # return batches.mean(axis=1).sort_values(ascending=False)

print(batch_node_means(data, 5, 682, 744))


In [ ]:
print(batch_node_means(data, 5, 682, 1200))


In [ ]:
def coverage_epochs_nodes(data, sampling_layers, batch_size, num_samples, epochs=300):
    batches_n_ids = {}

    for node_type in data.node_types:
        batches_n_ids[node_type] = set()

    loader = sampling({'sampling_layers': sampling_layers, 'batch_size': batch_size, 'num_samples': num_samples, 'seeded_type': 'disease'}, data)

    for epoch in range(epochs):
        if epoch % 50 == 0:
            print(f'Epoch {epoch}/{epochs}')

        for batch_id, batch in enumerate(iter(loader)):
            # batches_e_ids[batch_id] = {}
            # print(f'\nBatch {batch_id}: {batch.num_edges} edges')
            for node_type in batch.node_types:
                batches_n_ids[node_type].update(batch[node_type].n_id.tolist())
                    # batches_e_ids[batch_id][et].update(batch[et].e_id.tolist())
                    # print(f"{et[1]}: {len(set(batch[et].e_id.tolist()))}")

    print()

    for node_type in data.node_types:
        print(f'{node_type}: \t{len(batches_n_ids[node_type])} ({len(batches_n_ids[node_type]) / data[node_type].num_nodes * 100:.2f}%)')

    # return batches_n_ids

coverage_epochs_nodes(data, 5, 682, 744, epochs=300)


In [ ]:
coverage_epochs_nodes(data, 5, 682, 744, epochs=300)

In [ ]:
coverage_epochs_nodes(data, 5, 682, 1200, epochs=300)


In [ ]:
coverage_epochs_nodes(data, 5, 682, 1200, epochs=300)

In [ ]:
[1,2,3,4,5][:4]

In [ ]:
from torch_geometric.loader import HGTLoader

def sampling(config, data, workers = 0):
   num_samples = {
      # "drug": [config['num_samples']] * config['sampling_layers'],
      "drug": [500, 1000, 1500][:config['sampling_layers']],
      "disease": [config['num_samples']] * config['sampling_layers'],
      "gene/protein": [5200] * config['sampling_layers'],
      "effect/phenotype": [5200] * config['sampling_layers']
   }
   [config['num_samples']] * config['sampling_layers']
   loader = HGTLoader(data, 
                  # num_samples=[config['num_samples']] * config['sampling_layers'], 
                  num_samples=num_samples,
                  batch_size=config['batch_size'], 
               #    input_nodes=(config['node_type1'], data[config['node_type1']]['input_nodes']), 
                  input_nodes=(config['seeded_type'], data[config['seeded_type']].input_nodes), 
                  shuffle=True, 
                  num_workers = workers, 
                  persistent_workers = bool(workers)
                  )
   return loader


In [ ]:
coverage_epochs_nodes(data, 5, 682, 1200, epochs=300)

In [ ]:
s = coverage_epochs(data, 5, 682, 1200, epochs=300)

In [ ]:
s = coverage_epochs(data, 5, 682, 1200, epochs=500)

### probing by layer

In [ ]:
def node_counts(data):
    counts_nodes = {}
    counts_edges = {}
    # processed_edge_types = set()
    for node_type in data.node_types:
        # if node_type in processed_edge_types:
        #     continue
        # else:
        #     processed_edge_types.add(node_type)
            counts_nodes[node_type] = data[node_type].num_nodes

    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] in processed_edge_types:
            continue
        else:
            processed_edge_types.add(edge_type[1])
            counts_edges[edge_type[1]] = data[edge_type].num_edges


    return counts_nodes, counts_edges

node_counts(data)

In [ ]:
def batch_means(data, sampling_layers, batch_size, num_samples):
    config = {
        'sampling_layers': sampling_layers,
        'batch_size': batch_size,
        'num_samples': num_samples,
        'seeded_type': 'drug'
    }

    loader = sampling(config, data)
    batch_nodes_counts = []
    batch_edges_counts = []
    for batch in iter(loader):
        # print(batch.num_edges)
        batch_nodes_counts.append(node_counts(batch)[0])

        batch_edges_counts.append(node_counts(batch)[1])

    processed_edge_types = set()
    for edge_type in data.edge_types:
        if edge_type[1] in processed_edge_types:
            continue
        else:
            processed_edge_types.add(edge_type[1])
            # print(f'{edge_type[1]}: \t{data[edge_type].num_edges} edges')
            for batch_id in range(len(batch_edges_counts)):
                batch_edges_counts[batch_id][edge_type[1]] = round(batch_edges_counts[batch_id][edge_type[1]] / data[edge_type].num_edges * 100, 2)


    batches = pd.DataFrame(batch_nodes_counts).T
    print(batches)
    print(pd.DataFrame(batch_edges_counts).T)
    # return batches, pd.DataFrame(batch_edges_counts).T
    # batches = batches / batches.sum(axis=0) * 100
    # return batches.mean(axis=1).sort_values(ascending=False)

print(batch_means(data, 5, 682, 744))


In [ ]:
layers = [1, 2, 3, 4, 5]
for l in layers:
    print(f"Sampling layers: {l}")
    print(batch_means(data, l, 682, 1200), '\n')


### looking for least overlap (still representative) and full overlap

In [ ]:
print(batch_edge_means(data, 3, 682, 148) - batch_edge_means(data_pred, 3, 682, 148))

In [ ]:
abs(batch_edge_means(data, 3, 682, 148) - batch_edge_means(data_pred, 3, 682, 148)).sum()/2

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 3, 'batch_size': 682, 'num_samples': 148, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
1 - (71194 + 70326) / data.num_edges

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 3, 'batch_size': 682, 'num_samples': 744, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 4, 'batch_size': 682, 'num_samples': 596, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
print(exclusivity_report(sampling({'sampling_layers': 5, 'batch_size': 682, 'num_samples': 512, 'seeded_type': 'disease'}, data)).mean(axis=1))

In [ ]:
data.num_edges

In [ ]:
1 - (279639 + 278142) / data.num_edges, 1 - (284660 + 285128) / data.num_edges, 1 - (292338 + 291650) / data.num_edges